# 22.1 沒有標籤的兩種視圖

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/22-views/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""同一張圖的 global/local views；bbox 只用來看 crop，從不餵給 SSL。"""
import json
from pathlib import Path

import torch
import torch.nn.functional as F

from miniyolo.figures import save_svg, use_svg_text
from miniyolo.self_distillation import random_view
from miniyolo.vision_data import ColorRectangles


def main():
    torch.set_num_threads(2)
    data = ColorRectangles(samples=1, seed=101)
    image, bbox = data.images[:1], data.boxes[0]
    generator = torch.Generator().manual_seed(22)
    global_a, crop_a = random_view(image, generator)
    global_b, crop_b = random_view(image, generator)
    local, crop_local = random_view(image, generator, scale=(.15, .25))
    repeated_generator = torch.Generator().manual_seed(22)
    repeated = [random_view(image, repeated_generator, scale=scale)[0]
                for scale in ((.65, 1.), (.65, 1.), (.15, .25))]
    assert all(torch.equal(a, b) for a, b in zip((global_a, global_b, local), repeated))
    # 明確的反例：從四角選一個完全沒碰到物件的 8×8 crop。
    candidates = torch.tensor([[0, 0, 8, 8], [24, 0, 32, 8], [0, 24, 8, 32], [24, 24, 32, 32]])
    overlaps = (torch.minimum(candidates[:, 2:], bbox[2:]) -
                torch.maximum(candidates[:, :2], bbox[:2])).clamp_min(0).prod(-1)
    empty_crop = candidates[(overlaps == 0).nonzero()[0, 0]]
    x1, y1, x2, y2 = empty_crop.tolist()
    empty = F.interpolate(image[:, :, y1:y2, x1:x2], (32, 32), mode="bilinear", align_corners=False)
    result = {"source_seed": 101, "source_sample_index": 0, "source_samples": 1,
              "view_generator_seed": 22, "same_rng_reproduces_views": True,
              "input_shape": list(image.shape), "source_bbox_xyxy_pixel": bbox.tolist(),
              "global_a_crop_xyxy_pixel": crop_a[0].tolist(), "global_b_crop_xyxy_pixel": crop_b[0].tolist(),
              "local_crop_xyxy_pixel": crop_local[0].tolist(), "empty_crop_xyxy_pixel": empty_crop.tolist(),
              "empty_crop_object_intersection_pixel2": 0, "view_shape": list(global_a.shape),
              "ssl_inputs": "images only; labels/bboxes excluded",
              "role": "official DINO: teacher gets global; student gets global+local; later tiny trainer uses only 2 globals",
              "limitation": "same source does not guarantee a crop contains the object; views are not labeled detection examples"}
    output = Path("artifacts/runs/dino")
    output.mkdir(parents=True, exist_ok=True)
    use_svg_text()
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    figure, axes = plt.subplots(1, 5, figsize=(13, 3.7))
    panels = [(image, "原圖：bbox 只供說明", None), (global_a, "global A", crop_a[0]),
              (global_b, "global B", crop_b[0]), (local, "local 示範", crop_local[0]),
              (empty, "裁走物件的反例", empty_crop)]
    for axis, (view, name, crop) in zip(axes, panels):
        axis.imshow(view[0].permute(1, 2, 0).numpy(), vmin=0, vmax=1)
        axis.set_title(name)
        axis.axis("off")
        if crop is not None:
            axis.text(.5, -.05, f"原圖 crop={crop.tolist()}", transform=axis.transAxes, ha="center", fontsize=9)
    axes[0].add_patch(Rectangle((int(bbox[0]) - .5, int(bbox[1]) - .5),
                               int(bbox[2] - bbox[0]), int(bbox[3] - bbox[1]),
                               edgecolor="white", fill=False, linewidth=2))
    figure.suptitle("同一張圖片可形成不同 view；每個 crop 都 resize 回 32×32")
    figure.tight_layout()
    save_svg(figure, output / "22-views.svg", "實際 global 與 local views", "原圖、global A、global B、local 與裁走物件的 crop")
    plt.close(figure)
    (output / "22-views.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print(f"actual views: {output / '22-views.svg'}")


if __name__ == "__main__":
    main()


{
  "source_seed": 101,
  "source_sample_index": 0,
  "source_samples": 1,
  "view_generator_seed": 22,
  "same_rng_reproduces_views": true,
  "input_shape": [
    1,
    3,
    32,
    32
  ],
  "source_bbox_xyxy_pixel": [
    14,
    10,
    22,
    22
  ],
  "global_a_crop_xyxy_pixel": [
    1,
    1,
    29,
    29
  ],
  "global_b_crop_xyxy_pixel": [
    2,
    1,
    29,
    28
  ],
  "local_crop_xyxy_pixel": [
    4,
    11,
    19,
    26
  ],
  "empty_crop_xyxy_pixel": [
    0,
    0,
    8,
    8
  ],
  "empty_crop_object_intersection_pixel2": 0,
  "view_shape": [
    1,
    3,
    32,
    32
  ],
  "ssl_inputs": "images only; labels/bboxes excluded",
  "role": "official DINO: teacher gets global; student gets global+local; later tiny trainer uses only 2 globals",
  "limitation": "same source does not guarantee a crop contains the object; views are not labeled detection examples"
}
actual views: artifacts/runs/dino/22-views.svg
